# Preliminary Experiments: Qwen3.5-9B for Dermatological Diagnostics

This notebook evaluates the **Qwen3.5-9B** Vision Language Model on dermatological tasks as preliminary experiments for the dissertation's multi-agent diagnostic system.

**Experiments:**
1. Zero-shot text Q&A on dermatology questions
2. Zero-shot image classification on DermNet (23 classes)
3. RAG-style evaluation with synthetic knowledge base
4. LoRA fine-tuning on DermNet
5. Knowledge distillation from GPT-4o

**Hardware:** Google Colab T4 16GB GPU with 4-bit quantization

In [ ]:
%%capture
# Install dependencies for Qwen3.5 Vision on Colab T4
!pip install torch==2.8.0 torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install triton>=3.3.0 bitsandbytes xformers==0.0.32.post2
!pip install unsloth unsloth_zoo
!pip install transformers>=4.52.0 trl>=0.22.0
!pip install flash-linear-attention causal_conv1d==1.6.0
!pip install datasets scikit-learn matplotlib seaborn pandas kagglehub
!pip install openai  # for knowledge distillation with GPT-4o

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import os
import re
import random
import time
import difflib
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Configuration
MODEL_NAME = "unsloth/Qwen3.5-9B"
SEED = 3407
SAMPLE_SIZE = 200  # test subset for quick evaluation

DERMNET_CLASSES = [
    "Acne and Rosacea", "Actinic Keratosis", "Atopic Dermatitis",
    "Basal Cell Carcinoma", "Bullous Disease", "Cellulitis",
    "Eczema", "Exanthems and Drug Eruptions", "Hair Loss Alopecia",
    "Herpes HPV and STDs", "Light Diseases and Pigmentation",
    "Lupus and Connective Tissue", "Melanoma Skin Cancer Nevi",
    "Nail Fungus and Infection", "Poison Ivy and Contact Dermatitis",
    "Psoriasis Lichen Planus", "Scabies Lyme and Bites",
    "Seborrheic Keratoses", "Systemic Disease",
    "Tinea Ringworm Candidiasis", "Urticaria Hives",
    "Vascular Tumors", "Warts Molluscum and Viral"
]
NUM_CLASSES = len(DERMNET_CLASSES)

# Store results across all experiments
results = {}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Model Loading

**Qwen3.5-9B** is a Vision Language Model with:
- 9B parameters, hybrid architecture (Gated DeltaNet + Gated Attention)
- 262K native context window (extendable to 1M)
- Image, text, and video understanding
- Thinking mode (disabled here for direct answers)

We load with **4-bit NF4 quantization** (QLoRA) to fit within the T4's 16GB VRAM (~5-6GB model footprint).

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Model loaded successfully.")

In [ ]:
# GPU memory after model loading
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM allocated: {allocated:.2f} GB")
    print(f"VRAM reserved:  {reserved:.2f} GB")
    print(f"VRAM total:     {total:.1f} GB")
    print(f"VRAM free:      {total - reserved:.2f} GB")

## 3. DermNet Dataset

The **DermNet** dataset contains ~19,500 clinical photographs across 23 skin disease classes, sourced from the DermNet NZ dermatological atlas. We use stratified splitting (70/15/15) and subsample 200 test images for quick evaluation.

In [ ]:
import kagglehub

# Download DermNet dataset from Kaggle
# You may need to set up Kaggle credentials first:
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

dataset_path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print(f"Dataset downloaded to: {dataset_path}")

# Build DataFrame from directory structure
data_records = []
dataset_root = Path(dataset_path)

# DermNet on Kaggle has train/ and test/ subdirectories with class folders
for split_dir in ["train", "test"]:
    split_path = dataset_root / split_dir
    if not split_path.exists():
        continue
    for class_dir in sorted(split_path.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                data_records.append({
                    "image_path": str(img_file),
                    "label": class_name,
                    "original_split": split_dir
                })

df = pd.DataFrame(data_records)
print(f"Total images found: {len(df)}")
print(f"Unique classes: {df['label'].nunique()}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")

# Create label index mapping
label2idx = {label: idx for idx, label in enumerate(sorted(df["label"].unique()))}
idx2label = {idx: label for label, idx in label2idx.items()}
df["label_idx"] = df["label"].map(label2idx)

# Stratified split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_idx"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_idx"], random_state=SEED
)

print(f"\nSplit sizes - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Subsample test set for quick evaluation
test_subset = test_df.groupby("label").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_SIZE // NUM_CLASSES + 1), random_state=SEED)
).reset_index(drop=True)
test_subset = test_subset.sample(n=min(SAMPLE_SIZE, len(test_subset)), random_state=SEED).reset_index(drop=True)
print(f"Test subset for evaluation: {len(test_subset)} images")

In [ ]:
# Class distribution and sample images
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart of class distribution
class_counts = df["label"].value_counts().sort_index()
axes[0].barh(range(len(class_counts)), class_counts.values, color="steelblue")
axes[0].set_yticks(range(len(class_counts)))
axes[0].set_yticklabels(class_counts.index, fontsize=8)
axes[0].set_xlabel("Number of Images")
axes[0].set_title("DermNet Class Distribution")
axes[0].invert_yaxis()

axes[1].axis("off")
axes[1].set_title("Sample Images (see grid below)")
plt.tight_layout()
plt.show()

# Sample images grid
fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for idx, (label, group) in enumerate(df.groupby("label")):
    if idx >= 24:
        break
    ax = axes[idx // 6, idx % 6]
    try:
        img = Image.open(group.iloc[0]["image_path"]).convert("RGB")
        ax.imshow(img)
    except Exception:
        pass
    ax.set_title(label[:20], fontsize=7)
    ax.axis("off")
if NUM_CLASSES < 24:
    axes[3, 5].axis("off")
plt.suptitle("Sample Images per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Zero-Shot Text Q&A Baseline

Test the model's parametric dermatological knowledge before any fine-tuning or context injection.

In [ ]:
# Dermatology Q&A benchmark
QA_BENCHMARK = [
    {"question": "What are the primary symptoms of psoriasis?", "key_terms": ["plaques", "scaling", "erythema", "itching", "silvery", "skin"], "category": "factual"},
    {"question": "What causes tinea corporis?", "key_terms": ["dermatophyte", "fungal", "fungus", "ringworm", "trichophyton", "microsporum"], "category": "factual"},
    {"question": "What are the ABCDE criteria for melanoma detection?", "key_terms": ["asymmetry", "border", "color", "diameter", "evolving"], "category": "factual"},
    {"question": "What is the typical presentation of atopic dermatitis in adults?", "key_terms": ["eczema", "flexural", "pruritus", "dry", "chronic", "relapsing"], "category": "factual"},
    {"question": "What are the common treatments for acne vulgaris?", "key_terms": ["retinoid", "benzoyl peroxide", "antibiotic", "topical", "isotretinoin"], "category": "factual"},
    {"question": "How do you differentiate melanoma from seborrheic keratosis?", "key_terms": ["asymmetry", "border", "stuck-on", "benign", "biopsy", "dermoscopy"], "category": "differential"},
    {"question": "What distinguishes atopic dermatitis from contact dermatitis?", "key_terms": ["history", "allergen", "distribution", "chronic", "patch test", "atopy"], "category": "differential"},
    {"question": "How can you differentiate psoriasis from eczema clinically?", "key_terms": ["silvery", "well-defined", "extensor", "flexural", "scaling", "plaques"], "category": "differential"},
    {"question": "What are the key differences between basal cell carcinoma and squamous cell carcinoma?", "key_terms": ["nodular", "pearly", "ulceration", "keratinization", "metastasis", "sun"], "category": "differential"},
    {"question": "How do you distinguish urticaria from other erythematous conditions?", "key_terms": ["wheals", "transient", "pruritus", "hours", "histamine", "blanching"], "category": "differential"},
    {"question": "A patient presents with a rapidly growing, asymmetric pigmented lesion with irregular borders on the back. What is the most likely diagnosis and recommended next steps?", "key_terms": ["melanoma", "biopsy", "excision", "dermatologist", "urgent", "referral"], "category": "reasoning"},
    {"question": "A child presents with intensely itchy, linear burrows between the fingers. What is the likely diagnosis and treatment?", "key_terms": ["scabies", "permethrin", "mite", "burrows", "contacts", "treatment"], "category": "reasoning"},
    {"question": "A patient has multiple grouped vesicles on an erythematous base on the lip. What is the diagnosis and management?", "key_terms": ["herpes", "HSV", "simplex", "antiviral", "acyclovir", "valacyclovir"], "category": "reasoning"},
    {"question": "An elderly patient presents with a pearly, telangiectatic nodule on the nose that has been slowly growing. What is the most likely diagnosis?", "key_terms": ["basal cell carcinoma", "BCC", "nodular", "pearly", "biopsy", "excision"], "category": "reasoning"},
    {"question": "A patient develops widespread target lesions on the trunk and extremities after starting a new medication. What condition should be suspected?", "key_terms": ["erythema multiforme", "Stevens-Johnson", "drug reaction", "target", "discontinue"], "category": "reasoning"}
]

In [ ]:
# Run zero-shot text Q&A
FastVisionModel.for_inference(model)

def ask_question(question, system_prompt=None, enable_thinking=False):
    """Run text-only Q&A inference."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": [{"type": "text", "text": question}]})

    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=enable_thinking
    )
    inputs = tokenizer(input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=512, temperature=0.7, top_p=0.8, top_k=20, do_sample=True
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

qa_results_baseline = []
system_prompt = "You are a dermatology expert. Answer concisely and accurately."

for qa in tqdm(QA_BENCHMARK, desc="Text Q&A (zero-shot)"):
    start = time.time()
    response = ask_question(qa["question"], system_prompt=system_prompt)
    elapsed = time.time() - start
    qa_results_baseline.append({"question": qa["question"], "category": qa["category"], "response": response, "latency_s": elapsed})

print(f"Completed {len(qa_results_baseline)} questions.")

In [ ]:
# Score text Q&A by key-term coverage
def compute_key_term_score(response, key_terms):
    """Fraction of key terms found in the response (case-insensitive)."""
    response_lower = response.lower()
    hits = sum(1 for term in key_terms if term.lower() in response_lower)
    return hits / len(key_terms) if key_terms else 0.0

for i, qa in enumerate(QA_BENCHMARK):
    score = compute_key_term_score(qa_results_baseline[i]["response"], qa["key_terms"])
    qa_results_baseline[i]["key_term_score"] = score
    qa_results_baseline[i]["response_length"] = len(qa_results_baseline[i]["response"].split())

qa_df_baseline = pd.DataFrame(qa_results_baseline)
print("=== Zero-Shot Text Q&A Results ===")
print(f"\nOverall key-term score: {qa_df_baseline['key_term_score'].mean():.3f}")
print(f"\nBy category:")
print(qa_df_baseline.groupby("category")["key_term_score"].mean().to_string())
print(f"\nAvg response length: {qa_df_baseline['response_length'].mean():.0f} words")
print(f"Avg latency: {qa_df_baseline['latency_s'].mean():.1f}s")

results["qa_baseline"] = qa_df_baseline[["category", "key_term_score"]].groupby("category").mean().to_dict()

## 5. Zero-Shot Image Classification Baseline

Classify DermNet test images into one of 23 categories using structured JSON output.

In [ ]:
CLASSIFICATION_PROMPT = """Classify this clinical dermatology image into exactly one of the following 23 categories:

{classes}

Respond with ONLY a JSON object in this exact format: {{"diagnosis": "<category name>"}}
Use the exact category name from the list above.""".format(
    classes="\n".join(f"- {c}" for c in DERMNET_CLASSES)
)

def parse_diagnosis(response):
    """Parse model response to extract diagnosis, with fuzzy matching fallback."""
    try:
        match = re.search(r'\{[^}]*"diagnosis"\s*:\s*"([^"]+)"[^}]*\}', response)
        if match:
            diagnosis = match.group(1)
            if diagnosis in DERMNET_CLASSES:
                return diagnosis
            closest = difflib.get_close_matches(diagnosis, DERMNET_CLASSES, n=1, cutoff=0.5)
            if closest:
                return closest[0]
    except Exception:
        pass
    # Fallback: search for any class name in the response
    response_lower = response.lower()
    for cls in DERMNET_CLASSES:
        if cls.lower() in response_lower:
            return cls
    return "UNPARSEABLE"

def classify_image(image_path, extra_context=None):
    """Classify a single image. Optionally inject context."""
    image = Image.open(image_path).convert("RGB")
    user_content = [{"type": "image", "image": image}, {"type": "text", "text": CLASSIFICATION_PROMPT}]
    messages = []
    if extra_context:
        messages.append({"role": "system", "content": extra_context})
    messages.append({"role": "user", "content": user_content})

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=64, temperature=0.1, top_k=20, do_sample=True)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_diagnosis(response), response

print("Classification functions defined.")

In [ ]:
# Run zero-shot image classification on test subset
FastVisionModel.for_inference(model)

zs_predictions, zs_true_labels, zs_raw_responses, zs_latencies = [], [], [], []
parse_failures = 0

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc="Zero-shot classification"):
    start = time.time()
    try:
        pred, raw = classify_image(row["image_path"])
    except Exception as e:
        pred, raw = "UNPARSEABLE", str(e)
    elapsed = time.time() - start
    if pred == "UNPARSEABLE":
        parse_failures += 1
    zs_predictions.append(pred)
    zs_true_labels.append(row["label"])
    zs_raw_responses.append(raw)
    zs_latencies.append(elapsed)

print(f"\nCompleted. Parse failures: {parse_failures}/{len(test_subset)}")
print(f"Avg latency per image: {np.mean(zs_latencies):.2f}s")

In [ ]:
def compute_classification_metrics(y_true, y_pred, label=""):
    """Compute and display classification metrics."""
    valid_mask = [p != "UNPARSEABLE" for p in y_pred]
    y_true_valid = [t for t, v in zip(y_true, valid_mask) if v]
    y_pred_valid = [p for p, v in zip(y_pred, valid_mask) if v]
    if not y_true_valid:
        print("No valid predictions to evaluate.")
        return {}

    acc = accuracy_score(y_true_valid, y_pred_valid)
    macro_f1 = f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true_valid, y_pred_valid, average="weighted", zero_division=0)

    print(f"=== {label} Classification Metrics ===")
    print(f"Valid predictions: {len(y_true_valid)}/{len(y_true)}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")

    all_labels = sorted(set(y_true_valid + y_pred_valid))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=all_labels)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[l[:15] for l in all_labels],
                yticklabels=[l[:15] for l in all_labels], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix - {label}")
    plt.xticks(rotation=45, ha="right", fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout(); plt.show()

    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "valid_preds": len(y_true_valid), "total": len(y_true)}

results["zero_shot"] = compute_classification_metrics(zs_true_labels, zs_predictions, "Zero-Shot")

## 6. Synthetic Knowledge Base & RAG-Style Evaluation

We create a **template-based synthetic dermatology knowledge base** with entries for each of the 23 DermNet classes, then inject it as context before the model answers. This simulates RAG without requiring ChromaDB/BGE-M3 infrastructure.

In [ ]:
KNOWLEDGE_BASE = {
    "Acne and Rosacea": {"description": "Acne vulgaris is a chronic inflammatory condition of the pilosebaceous unit, characterized by comedones, papules, pustules, nodules, and cysts. Rosacea is a chronic inflammatory dermatosis affecting the central face with persistent erythema, telangiectasia, papules, and pustules.", "key_features": "Comedones (open/closed), inflammatory papules and pustules on face/trunk/shoulders. Rosacea: central facial erythema, flushing, telangiectasia, rhinophyma. Rosacea lacks comedones.", "differential_diagnosis": "Perioral dermatitis, folliculitis, seborrheic dermatitis, lupus malar rash", "typical_presentation": "Adolescents and young adults (acne); adults 30-50, fair-skinned (rosacea)"},
    "Actinic Keratosis": {"description": "Precancerous epidermal lesions from cumulative UV exposure. 5-10% risk of progression to invasive SCC.", "key_features": "Rough, scaly, erythematous macules/papules on sun-exposed areas. Gritty sandpaper-like texture. May have cutaneous horn.", "differential_diagnosis": "SCC in situ (Bowen's disease), seborrheic keratosis, superficial BCC, psoriasis", "typical_presentation": "Elderly, fair skin, chronic sun exposure"},
    "Atopic Dermatitis": {"description": "Chronic, relapsing inflammatory skin condition, most common eczema form. Part of atopic triad with asthma and allergic rhinitis.", "key_features": "Intense pruritus, xerosis, erythematous patches with excoriations. Flexural distribution in adults. Lichenification in chronic disease.", "differential_diagnosis": "Contact dermatitis, seborrheic dermatitis, psoriasis, scabies", "typical_presentation": "Onset in infancy/childhood, family history of atopy"},
    "Basal Cell Carcinoma": {"description": "Most common human malignancy from epidermal basal cells. Locally invasive, rarely metastasizes (<0.1%). UV exposure is primary risk factor.", "key_features": "Pearly/translucent papule/nodule with rolled borders and telangiectasia. May ulcerate centrally (rodent ulcer). Subtypes: nodular, superficial, morpheaform.", "differential_diagnosis": "SCC, melanoma (pigmented BCC), intradermal nevus, sebaceous hyperplasia", "typical_presentation": "Older adults, sun-exposed areas (face/nose), fair skin"},
    "Bullous Disease": {"description": "Autoimmune conditions with blister formation. Bullous pemphigoid (subepidermal, tense) and pemphigus vulgaris (intraepidermal, flaccid).", "key_features": "Tense or flaccid blisters. Nikolsky sign positive in pemphigus. BP: large tense bullae on trunk/limbs. PV: flaccid blisters, oral erosions.", "differential_diagnosis": "Dermatitis herpetiformis, linear IgA disease, SJS, contact dermatitis", "typical_presentation": "BP in elderly (>60); PV in 40-60 year olds"},
    "Cellulitis": {"description": "Acute spreading bacterial infection of dermis and subcutaneous tissue. Usually Streptococcus or Staphylococcus.", "key_features": "Expanding erythema, warmth, edema, tenderness. Poorly defined borders. May have fever, lymphangitic streaking.", "differential_diagnosis": "DVT, erysipelas, stasis dermatitis, necrotizing fasciitis", "typical_presentation": "Lower extremities; risk factors: skin breaks, lymphedema, obesity, diabetes"},
    "Eczema": {"description": "Group of inflammatory conditions including nummular, dyshidrotic, and asteatotic eczema. Disrupted skin barrier.", "key_features": "Erythema, vesicles (acute), scaling/lichenification (chronic), intense pruritus. Nummular: coin-shaped. Dyshidrotic: palm/sole vesicles.", "differential_diagnosis": "Psoriasis, tinea, contact dermatitis, atopic dermatitis", "typical_presentation": "Variable age; nummular in middle-aged men; dyshidrotic in young adults"},
    "Exanthems and Drug Eruptions": {"description": "Widespread rashes from viral or drug causes. Drug eruptions range from morbilliform to SJS/TEN.", "key_features": "Morbilliform: symmetric erythematous macules/papules from trunk to extremities. 7-14 days after drug. SJS/TEN: target lesions, mucosal erosions.", "differential_diagnosis": "Viral exanthem, secondary syphilis, Kawasaki disease, scarlet fever", "typical_presentation": "Temporal relationship with medication or viral illness"},
    "Hair Loss Alopecia": {"description": "Hair loss classified as scarring or non-scarring. Types: androgenetic, alopecia areata, telogen effluvium, traction.", "key_features": "Alopecia areata: smooth round patches, exclamation point hairs. Androgenetic: progressive thinning at temples/vertex. Telogen effluvium: diffuse shedding.", "differential_diagnosis": "Tinea capitis, trichotillomania, secondary syphilis, thyroid disease", "typical_presentation": "Alopecia areata: any age; androgenetic: progressive with age"},
    "Herpes HPV and STDs": {"description": "Cutaneous STI manifestations: HSV (oral/genital herpes), HPV (genital warts), syphilis.", "key_features": "HSV: grouped vesicles on erythematous base. Genital warts: flesh-colored verrucous papules. Syphilis: painless chancre (primary), palms/soles rash (secondary).", "differential_diagnosis": "Aphthous ulcers, Behcet's, molluscum contagiosum, lichen planus", "typical_presentation": "Sexually active adults; HSV recurrences from stress/UV/immunosuppression"},
    "Light Diseases and Pigmentation": {"description": "Pigmentation disorders: vitiligo (depigmentation), melasma (hyperpigmentation), photodermatoses.", "key_features": "Vitiligo: well-demarcated depigmented patches, symmetric. Melasma: brown patches on face. PMLE: pruritic papules on sun-exposed areas.", "differential_diagnosis": "Post-inflammatory changes, tinea versicolor, pityriasis alba, lupus", "typical_presentation": "Vitiligo: any age; melasma: childbearing women; photodermatoses: spring/summer"},
    "Lupus and Connective Tissue": {"description": "Cutaneous lupus (malar rash, discoid) and dermatomyositis (heliotrope rash, Gottron papules).", "key_features": "SLE: butterfly malar rash sparing nasolabial folds. Discoid: erythematous plaques with scarring. Dermatomyositis: violaceous periorbital edema, Gottron papules.", "differential_diagnosis": "Rosacea, seborrheic dermatitis, PMLE, psoriasis", "typical_presentation": "SLE: young women 15-45; discoid: women 20-40; dermatomyositis: bimodal"},
    "Melanoma Skin Cancer Nevi": {"description": "Malignant melanocyte neoplasm, most lethal skin cancer. Nevi are benign precursors. ABCDE criteria guide detection.", "key_features": "ABCDE: Asymmetry, Border irregularity, Color variation, Diameter >6mm, Evolving. Subtypes: superficial spreading, nodular, lentigo maligna, acral lentiginous.", "differential_diagnosis": "Dysplastic nevus, seborrheic keratosis, pigmented BCC, dermatofibroma", "typical_presentation": "Fair-skinned adults; acral lentiginous in darker skin on palms/soles"},
    "Nail Fungus and Infection": {"description": "Onychomycosis from dermatophytes (T. rubrum), yeasts, or molds. Most common nail disorder.", "key_features": "DLSO: yellow-white discoloration at distal edge, subungual hyperkeratosis, onycholysis, thickening. WSO: white crumbly patches. PSO: white near cuticle.", "differential_diagnosis": "Psoriatic nails, nail trauma, lichen planus, subungual melanoma", "typical_presentation": "Adults, prevalence increases with age; diabetes, PVD, immunosuppression"},
    "Poison Ivy and Contact Dermatitis": {"description": "Inflammatory reaction from skin contact with irritant or allergen. Allergic CD is type IV hypersensitivity.", "key_features": "Poison ivy: linear vesicle/bullae streaks, intense pruritus. Allergic CD: well-demarcated erythema matching contact pattern. Irritant: dryness, fissuring.", "differential_diagnosis": "Atopic dermatitis, nummular eczema, herpes zoster, bullous pemphigoid", "typical_presentation": "Distribution matches contactant exposure"},
    "Psoriasis Lichen Planus": {"description": "Psoriasis: chronic autoimmune with epidermal hyperproliferation. Lichen planus: 5 Ps (Pruritic, Purple, Polygonal, Planar, Papules).", "key_features": "Psoriasis: well-demarcated plaques with silvery scales on extensors, Auspitz sign, nail pitting. LP: violaceous papules with Wickham striae, oral involvement.", "differential_diagnosis": "Eczema, seborrheic dermatitis, pityriasis rosea, secondary syphilis", "typical_presentation": "Psoriasis: bimodal 20-30/50-60; LP: adults 30-60"},
    "Scabies Lyme and Bites": {"description": "Scabies: Sarcoptes scabiei mite. Lyme: Borrelia via Ixodes tick. Insect bites: localized reactions.", "key_features": "Scabies: intense nocturnal pruritus, burrows in web spaces/wrists/genitalia. Lyme: erythema migrans (bull's-eye). Bites: grouped pruritic papules.", "differential_diagnosis": "Atopic dermatitis, contact dermatitis, folliculitis, urticaria", "typical_presentation": "Scabies: close contacts; Lyme: endemic areas, outdoor exposure"},
    "Seborrheic Keratoses": {"description": "Most common benign epidermal tumors from keratinocytes. No malignant potential.", "key_features": "Well-demarcated, waxy, stuck-on appearance. Tan to dark brown/black. Horn cysts and fissures. Leser-Trelat sign: sudden multiple SKs may indicate malignancy.", "differential_diagnosis": "Melanoma, pigmented BCC, wart, actinic keratosis", "typical_presentation": "Adults over 30, trunk and face, familial tendency"},
    "Systemic Disease": {"description": "Cutaneous manifestations of systemic disease: xanthomas, necrobiosis lipoidica, acanthosis nigricans, pyoderma gangrenosum.", "key_features": "Acanthosis nigricans: velvety hyperpigmented plaques in skin folds. Necrobiosis lipoidica: yellow-brown atrophic shin plaques. Pyoderma gangrenosum: painful ulcers with violaceous borders.", "differential_diagnosis": "Primary dermatological conditions, fungal infections, vasculitis", "typical_presentation": "Varies; acanthosis nigricans in obese/diabetic; necrobiosis in T1DM"},
    "Tinea Ringworm Candidiasis": {"description": "Superficial fungal infections by dermatophytes (tinea) or Candida. Named by body site.", "key_features": "Tinea corporis: annular plaques with raised scaly border and central clearing. Tinea pedis: interdigital maceration. Candidiasis: beefy red erythema with satellite pustules.", "differential_diagnosis": "Eczema, psoriasis, contact dermatitis, pityriasis rosea, granuloma annulare", "typical_presentation": "Any age; tinea pedis in athletes; candidiasis in immunocompromised"},
    "Urticaria Hives": {"description": "Transient pruritic wheals from mast cell degranulation. Acute (<6wk) often allergic; chronic (>6wk) often autoimmune/idiopathic.", "key_features": "Wheals: raised erythematous plaques that blanch, last <24h. Intense pruritus. Angioedema may accompany. Dermatographism in some.", "differential_diagnosis": "Urticarial vasculitis, erythema multiforme, drug eruption, mastocytosis", "typical_presentation": "Any age; acute from foods/meds/infections; chronic in adults 20-40"},
    "Vascular Tumors": {"description": "Benign (infantile hemangiomas, cherry angiomas, pyogenic granulomas) and malignant (Kaposi sarcoma, angiosarcoma).", "key_features": "Infantile hemangioma: bright red compressible nodule in neonates. Cherry angioma: red papules in adults. Pyogenic granuloma: rapidly growing friable nodule. Kaposi: purple-red in immunocompromised.", "differential_diagnosis": "Amelanotic melanoma, BCC, dermatofibroma, bacillary angiomatosis", "typical_presentation": "Hemangiomas in neonates; cherry angiomas increase with age; Kaposi in HIV"},
    "Warts Molluscum and Viral": {"description": "Common warts (HPV) and molluscum contagiosum (poxvirus). Benign, typically self-limited.", "key_features": "Warts: firm rough hyperkeratotic papules with black dots. Plantar: endophytic, painful on lateral compression. Molluscum: dome-shaped pearly umbilicated papules 2-5mm.", "differential_diagnosis": "Seborrheic keratosis, SCC (elderly), corn/callus, BCC", "typical_presentation": "Warts: children/young adults; molluscum: children 1-10, sexually active adults"}
}

print(f"Knowledge base created with {len(KNOWLEDGE_BASE)} entries.")

In [ ]:
def format_knowledge_base(kb):
    """Format the knowledge base as a context string."""
    sections = []
    for condition, info in kb.items():
        section = f"### {condition}\n{info['description']}\nKey Features: {info['key_features']}\nDifferentials: {info['differential_diagnosis']}\nTypical Presentation: {info['typical_presentation']}"
        sections.append(section)
    return "\n\n".join(sections)

RAG_CONTEXT = format_knowledge_base(KNOWLEDGE_BASE)
RAG_SYSTEM_PROMPT = f"""You are a dermatology expert. Use the following dermatological reference to inform your answers. Cite relevant information when applicable.

--- DERMATOLOGY REFERENCE ---
{RAG_CONTEXT}
--- END REFERENCE ---"""

print(f"Context length: ~{len(RAG_CONTEXT) // 4} tokens")

In [ ]:
# RAG-style image classification
FastVisionModel.for_inference(model)

rag_predictions, rag_true_labels, rag_latencies = [], [], []
rag_parse_failures = 0

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc="RAG classification"):
    start = time.time()
    try:
        pred, raw = classify_image(row["image_path"], extra_context=RAG_SYSTEM_PROMPT)
    except Exception:
        pred = "UNPARSEABLE"
    elapsed = time.time() - start
    if pred == "UNPARSEABLE":
        rag_parse_failures += 1
    rag_predictions.append(pred)
    rag_true_labels.append(row["label"])
    rag_latencies.append(elapsed)

print(f"\nParse failures: {rag_parse_failures}/{len(test_subset)}")
print(f"Avg latency: {np.mean(rag_latencies):.2f}s")

In [ ]:
# RAG-style text Q&A
qa_results_rag = []
for qa in tqdm(QA_BENCHMARK, desc="Text Q&A (RAG)"):
    start = time.time()
    response = ask_question(qa["question"], system_prompt=RAG_SYSTEM_PROMPT)
    elapsed = time.time() - start
    score = compute_key_term_score(response, qa["key_terms"])
    qa_results_rag.append({"question": qa["question"], "category": qa["category"], "response": response, "key_term_score": score, "response_length": len(response.split()), "latency_s": elapsed})

qa_df_rag = pd.DataFrame(qa_results_rag)
print(f"\nRAG Q&A key-term score: {qa_df_rag['key_term_score'].mean():.3f}")
print(f"Baseline Q&A key-term score: {qa_df_baseline['key_term_score'].mean():.3f}")

In [ ]:
# Compare zero-shot vs RAG
results["rag"] = compute_classification_metrics(rag_true_labels, rag_predictions, "RAG-Augmented")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_names = ["accuracy", "macro_f1", "weighted_f1"]
x = np.arange(len(metrics_names))
width = 0.35
zs_vals = [results["zero_shot"].get(m, 0) for m in metrics_names]
rag_vals = [results["rag"].get(m, 0) for m in metrics_names]

axes[0].bar(x - width/2, zs_vals, width, label="Zero-Shot", color="steelblue")
axes[0].bar(x + width/2, rag_vals, width, label="RAG-Augmented", color="coral")
axes[0].set_xticks(x); axes[0].set_xticklabels(["Accuracy", "Macro-F1", "Weighted-F1"])
axes[0].set_ylabel("Score"); axes[0].set_title("Image Classification: Zero-Shot vs RAG")
axes[0].legend(); axes[0].set_ylim(0, 1)

categories = ["factual", "differential", "reasoning"]
bl_scores = [qa_df_baseline[qa_df_baseline["category"] == c]["key_term_score"].mean() for c in categories]
rg_scores = [qa_df_rag[qa_df_rag["category"] == c]["key_term_score"].mean() for c in categories]
x2 = np.arange(len(categories))
axes[1].bar(x2 - width/2, bl_scores, width, label="Zero-Shot", color="steelblue")
axes[1].bar(x2 + width/2, rg_scores, width, label="RAG-Augmented", color="coral")
axes[1].set_xticks(x2); axes[1].set_xticklabels(["Factual", "Differential", "Reasoning"])
axes[1].set_ylabel("Key-Term Score"); axes[1].set_title("Text Q&A: Zero-Shot vs RAG")
axes[1].legend(); axes[1].set_ylim(0, 1)

plt.tight_layout(); plt.show()

## 7. LoRA Fine-Tuning on DermNet

Configuration matches Chapter 3: Rank=16, Alpha=16, Dropout=0.05, language attention only, LR=2e-5, cosine+warmup(5%), AdamW, FP16.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,
    r=16, lora_alpha=16, lora_dropout=0.05,
    bias="none", random_state=SEED, use_rslora=False, loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
TRAIN_SUBSET_SIZE = 3000
train_sample = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)
val_sample = val_df.sample(n=min(500, len(val_df)), random_state=SEED).reset_index(drop=True)
print(f"Training subset: {len(train_sample)}, Validation subset: {len(val_sample)}")

TRAIN_PROMPT = "Classify this clinical dermatology image into one of 23 categories: {classes}. Respond with JSON: {{\"diagnosis\": \"<category>\"}}".format(classes=", ".join(DERMNET_CLASSES))

def create_conversation(image_path, label):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": Image.open(image_path).convert("RGB")}, {"type": "text", "text": TRAIN_PROMPT}]},
        {"role": "assistant", "content": [{"type": "text", "text": json.dumps({"diagnosis": label})}]}
    ]}

print("Converting training data...")
train_conversations = []
for _, row in tqdm(train_sample.iterrows(), total=len(train_sample), desc="Train"):
    try: train_conversations.append(create_conversation(row["image_path"], row["label"]))
    except: continue

val_conversations = []
for _, row in tqdm(val_sample.iterrows(), total=len(val_sample), desc="Val"):
    try: val_conversations.append(create_conversation(row["image_path"], row["label"]))
    except: continue

print(f"Train: {len(train_conversations)}, Val: {len(val_conversations)}")

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conversations, eval_dataset=val_conversations,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=16,
        warmup_ratio=0.05, num_train_epochs=3, learning_rate=2e-5,
        logging_steps=10, eval_steps=50, eval_strategy="steps",
        save_strategy="steps", save_steps=50, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="cosine", seed=SEED,
        output_dir="outputs_lora", report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048, fp16=True, bf16=False,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
    ),
)
print("Trainer configured.")

In [ ]:
if torch.cuda.is_available():
    print(f"VRAM before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"Training time: {trainer_stats.metrics.get('train_runtime', 0) / 60:.1f} min")
print(f"Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")

log_history = trainer.state.log_history
train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(10, 5))
if train_losses:
    s, l = zip(*train_losses); ax.plot(s, l, label="Train Loss", alpha=0.8)
if eval_losses:
    s, l = zip(*eval_losses); ax.plot(s, l, label="Eval Loss", marker="o", markersize=4)
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Training Loss Curve")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
model.save_pretrained("qwen35_9b_dermnet_lora")
tokenizer.save_pretrained("qwen35_9b_dermnet_lora")
print("LoRA adapters saved.")

## 8. Post-Fine-Tuning Evaluation

In [ ]:
FastVisionModel.for_inference(model)

ft_predictions, ft_true_labels, ft_latencies = [], [], []
ft_parse_failures = 0

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc="Fine-tuned classification"):
    start = time.time()
    try: pred, raw = classify_image(row["image_path"])
    except: pred = "UNPARSEABLE"
    elapsed = time.time() - start
    if pred == "UNPARSEABLE": ft_parse_failures += 1
    ft_predictions.append(pred)
    ft_true_labels.append(row["label"])
    ft_latencies.append(elapsed)

print(f"\nParse failures: {ft_parse_failures}/{len(test_subset)}")
print(f"Avg latency: {np.mean(ft_latencies):.2f}s")

In [ ]:
qa_results_ft = []
for qa in tqdm(QA_BENCHMARK, desc="Text Q&A (fine-tuned)"):
    start = time.time()
    response = ask_question(qa["question"], system_prompt="You are a dermatology expert. Answer concisely and accurately.")
    elapsed = time.time() - start
    score = compute_key_term_score(response, qa["key_terms"])
    qa_results_ft.append({"question": qa["question"], "category": qa["category"], "response": response, "key_term_score": score, "response_length": len(response.split()), "latency_s": elapsed})

qa_df_ft = pd.DataFrame(qa_results_ft)
print(f"\nFine-tuned Q&A key-term score: {qa_df_ft['key_term_score'].mean():.3f}")

In [ ]:
results["finetuned"] = compute_classification_metrics(ft_true_labels, ft_predictions, "Fine-Tuned")

In [ ]:
def per_class_f1(y_true, y_pred, classes):
    valid_mask = [p != "UNPARSEABLE" for p in y_pred]
    yt = [t for t, v in zip(y_true, valid_mask) if v]
    yp = [p for p, v in zip(y_pred, valid_mask) if v]
    _, _, f1s, _ = precision_recall_fscore_support(yt, yp, labels=classes, zero_division=0)
    return dict(zip(classes, f1s))

test_classes = sorted(test_subset["label"].unique())
zs_f1 = per_class_f1(zs_true_labels, zs_predictions, test_classes)
ft_f1 = per_class_f1(ft_true_labels, ft_predictions, test_classes)
deltas = {c: ft_f1.get(c, 0) - zs_f1.get(c, 0) for c in test_classes}
sorted_classes = sorted(deltas.keys(), key=lambda c: deltas[c])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["green" if deltas[c] >= 0 else "red" for c in sorted_classes]
ax.barh(range(len(sorted_classes)), [deltas[c] for c in sorted_classes], color=colors)
ax.set_yticks(range(len(sorted_classes)))
ax.set_yticklabels([c[:25] for c in sorted_classes], fontsize=8)
ax.set_xlabel("F1 Change (Fine-tuned - Zero-shot)")
ax.set_title("Per-Class F1 Improvement from LoRA Fine-Tuning")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## 9. Knowledge Distillation from GPT-4o

**Response-based distillation**: GPT-4o generates classification + explanations for training images, then we fine-tune a fresh Qwen3.5-9B on those outputs.

In [ ]:
import base64
from openai import OpenAI

# Set API key: os.environ["OPENAI_API_KEY"] = "sk-..."
# In Colab: from google.colab import userdata; os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

DISTILL_SIZE = 500
distill_sample = train_df.sample(n=min(DISTILL_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)

TEACHER_PROMPT = f"""Classify this clinical dermatology image into one of these 23 categories:
{', '.join(DERMNET_CLASSES)}

Respond with JSON: {{"diagnosis": "<exact category name>", "explanation": "<2-3 sentence clinical reasoning>"}}"""

def get_teacher_response(image_path):
    with open(image_path, "rb") as f:
        b64_image = base64.b64encode(f.read()).decode()
    ext = Path(image_path).suffix.lower()
    media_type = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}.get(ext, "image/jpeg")
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{b64_image}"}},
            {"type": "text", "text": TEACHER_PROMPT}
        ]}],
        max_tokens=256, temperature=0.3
    )
    raw = response.choices[0].message.content
    match = re.search(r'\{[^}]*"diagnosis"\s*:[^}]+\}', raw, re.DOTALL)
    if match:
        return json.loads(match.group())
    return {"diagnosis": "UNPARSEABLE", "explanation": raw}

teacher_labels = []
teacher_errors = 0
for _, row in tqdm(distill_sample.iterrows(), total=len(distill_sample), desc="GPT-4o labeling"):
    try:
        result = get_teacher_response(row["image_path"])
        teacher_labels.append({"image_path": row["image_path"], "ground_truth": row["label"],
                              "teacher_diagnosis": result.get("diagnosis", "UNPARSEABLE"),
                              "teacher_explanation": result.get("explanation", "")})
    except:
        teacher_errors += 1

teacher_df = pd.DataFrame(teacher_labels)
print(f"\nTeacher labels: {len(teacher_df)}, Errors: {teacher_errors}")
print(f"GPT-4o agreement with ground truth: {(teacher_df['teacher_diagnosis'] == teacher_df['ground_truth']).mean():.3f}")

In [ ]:
def create_distillation_conversation(image_path, teacher_output):
    response_text = json.dumps({"diagnosis": teacher_output["teacher_diagnosis"], "explanation": teacher_output["teacher_explanation"]})
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": Image.open(image_path).convert("RGB")},
                                     {"type": "text", "text": "Classify this skin condition and explain your reasoning. Respond as JSON."}]},
        {"role": "assistant", "content": [{"type": "text", "text": response_text}]}
    ]}

distill_conversations = []
for _, row in tqdm(teacher_df.iterrows(), total=len(teacher_df), desc="Preparing distillation data"):
    if row["teacher_diagnosis"] == "UNPARSEABLE": continue
    try: distill_conversations.append(create_distillation_conversation(row["image_path"], row))
    except: continue

print(f"Distillation conversations: {len(distill_conversations)}")

In [ ]:
# Reload fresh base model for distillation
del model, trainer
torch.cuda.empty_cache()
import gc; gc.collect()

model, tokenizer = FastVisionModel.from_pretrained(MODEL_NAME, load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=False,
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none", random_state=SEED,
    use_rslora=False, loftq_config=None,
)
print("Fresh model loaded for distillation.")
model.print_trainable_parameters()

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

distill_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=distill_conversations,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=16,
        warmup_ratio=0.05, num_train_epochs=3, learning_rate=2e-5,
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=SEED, output_dir="outputs_distill",
        report_to="none", remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048, fp16=True, bf16=False,
    ),
)
distill_stats = distill_trainer.train()
print(f"\nDistillation time: {distill_stats.metrics.get('train_runtime', 0) / 60:.1f} min")
print(f"Final loss: {distill_stats.metrics.get('train_loss', 'N/A')}")

In [ ]:
FastVisionModel.for_inference(model)

dist_predictions, dist_true_labels = [], []
dist_parse_failures = 0

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc="Distilled classification"):
    try: pred, _ = classify_image(row["image_path"])
    except: pred = "UNPARSEABLE"
    if pred == "UNPARSEABLE": dist_parse_failures += 1
    dist_predictions.append(pred)
    dist_true_labels.append(row["label"])

print(f"\nParse failures: {dist_parse_failures}/{len(test_subset)}")
results["distilled"] = compute_classification_metrics(dist_true_labels, dist_predictions, "Distilled")

In [ ]:
print("=== Distillation vs Direct Fine-Tuning ===")
print(pd.DataFrame({
    "Method": ["Direct Fine-Tuning (GT)", "Distillation (GPT-4o)"],
    "Train Size": [len(train_conversations), len(distill_conversations)],
    "Accuracy": [results["finetuned"].get("accuracy", 0), results["distilled"].get("accuracy", 0)],
    "Macro-F1": [results["finetuned"].get("macro_f1", 0), results["distilled"].get("macro_f1", 0)],
    "Weighted-F1": [results["finetuned"].get("weighted_f1", 0), results["distilled"].get("weighted_f1", 0)],
}).to_string(index=False))

## 10. Results Summary

In [ ]:
summary_df = pd.DataFrame({
    "Method": ["Zero-Shot", "RAG-Augmented", "LoRA Fine-Tuned", "Distilled (GPT-4o)"],
    "Accuracy": [results.get(k, {}).get("accuracy", 0) for k in ["zero_shot", "rag", "finetuned", "distilled"]],
    "Macro-F1": [results.get(k, {}).get("macro_f1", 0) for k in ["zero_shot", "rag", "finetuned", "distilled"]],
    "Weighted-F1": [results.get(k, {}).get("weighted_f1", 0) for k in ["zero_shot", "rag", "finetuned", "distilled"]],
})
print("=== Comprehensive Results ===")
print(summary_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(summary_df)); width = 0.25
ax.bar(x - width, summary_df["Accuracy"], width, label="Accuracy", color="steelblue")
ax.bar(x, summary_df["Macro-F1"], width, label="Macro-F1", color="coral")
ax.bar(x + width, summary_df["Weighted-F1"], width, label="Weighted-F1", color="seagreen")
ax.set_xticks(x); ax.set_xticklabels(summary_df["Method"], rotation=15, ha="right")
ax.set_ylabel("Score"); ax.set_title("Classification Performance Across Methods")
ax.legend(); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3, axis="y")
for bars in ax.containers: ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
plt.tight_layout(); plt.show()

In [ ]:
all_f1 = {
    "Zero-Shot": per_class_f1(zs_true_labels, zs_predictions, test_classes),
    "RAG": per_class_f1(rag_true_labels, rag_predictions, test_classes),
    "Fine-Tuned": per_class_f1(ft_true_labels, ft_predictions, test_classes),
    "Distilled": per_class_f1(dist_true_labels, dist_predictions, test_classes),
}
heatmap_df = pd.DataFrame(all_f1, index=test_classes)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1,
            yticklabels=[c[:25] for c in test_classes], ax=ax)
ax.set_title("Per-Class F1 Scores Across Methods"); ax.set_ylabel("Disease Category")
plt.tight_layout(); plt.show()

In [ ]:
qa_comparison = pd.DataFrame({
    "Category": ["factual", "differential", "reasoning", "OVERALL"],
    "Zero-Shot": [qa_df_baseline[qa_df_baseline["category"]==c]["key_term_score"].mean() for c in ["factual","differential","reasoning"]] + [qa_df_baseline["key_term_score"].mean()],
    "RAG": [qa_df_rag[qa_df_rag["category"]==c]["key_term_score"].mean() for c in ["factual","differential","reasoning"]] + [qa_df_rag["key_term_score"].mean()],
    "Fine-Tuned": [qa_df_ft[qa_df_ft["category"]==c]["key_term_score"].mean() for c in ["factual","differential","reasoning"]] + [qa_df_ft["key_term_score"].mean()],
})
print("=== Text Q&A Key-Term Scores ===")
print(qa_comparison.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
print("=== Resource Usage Summary ===")
print(f"Model: {MODEL_NAME}")
print(f"Quantization: 4-bit NF4 (QLoRA)")
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"\nInference Latency (per image):")
print(f"  Zero-shot:  {np.mean(zs_latencies):.2f}s (median: {np.median(zs_latencies):.2f}s)")
print(f"  RAG:        {np.mean(rag_latencies):.2f}s (median: {np.median(rag_latencies):.2f}s)")
print(f"  Fine-tuned: {np.mean(ft_latencies):.2f}s (median: {np.median(ft_latencies):.2f}s)")
print(f"\nTraining:")
print(f"  LoRA fine-tuning: {len(train_conversations)} samples")
print(f"  Distillation: {len(distill_conversations)} teacher-labeled samples")